# Feature Selection


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))

from src.config import (
    ARTIFACTS_DIR,
    ENGINEERED_DATA_FILE,
    FEATURE_COLUMNS_FILE,
    PROJECT_ROOT,
    TARGET_COLUMN,
)
from src.data.data_loader import load_engineered_data

NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / "10_feature_selection"
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "10_feature_selection"
SELECTED_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "creditcard_selected_features.csv"
SELECTED_FEATURES_FILE = ARTIFACTS_DIR / "selected_feature_names.json"

NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

print("Imports OK")
print(f"  Engineered data : {ENGINEERED_DATA_FILE}")
print(f"  Feature list    : {FEATURE_COLUMNS_FILE}")
print(f"  Tables dir      : {NOTEBOOK_TABLES_DIR}")
print(f"  Figures dir     : {NOTEBOOK_FIGURES_DIR}")
print(f"  Selected data   : {SELECTED_DATA_FILE}")
print(f"  Feature export  : {SELECTED_FEATURES_FILE}")


Imports OK
  Engineered data : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/creditcard_engineered.csv
  Feature list    : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/feature_columns.json
  Tables dir      : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/10_feature_selection
  Figures dir     : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/10_feature_selection
  Selected data   : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/creditcard_selected_features.csv
  Feature export  : /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/selected_feature_names.json


## 1. Objectives

- Measure how strongly each engineered feature relates to fraud.
- Detect redundant features that may duplicate the same signal.
- Add lightweight model-based evidence before deciding what moves into modeling.
- Build a final selected feature set with clear keep or drop decisions.
- Save tables and artifacts so the modeling notebook can start from a stable input set.


## Output Guide

This notebook writes its main outputs to:

- `reports/tables/10_feature_selection/`
- `reports/figures/10_feature_selection/`
- `data/processed/creditcard_selected_features.csv`
- `artifacts/selected_feature_names.json`

The final answer from this notebook is a feature-level decision table and a modeling-ready selected dataset.


## 2. Load Engineered Data


In [2]:
df = load_engineered_data(ENGINEERED_DATA_FILE)
print(f"Engineered dataset shape: {df.shape}")
df.head()


Engineered dataset shape: (283726, 23)


,V17,V14,V12,V10,V16,V3,V7,V11,V4,V18,...,time_day_fraction,amount_to_mean_ratio,amount_to_median_ratio,amount_V17_interaction,V17_V14_interaction,V17_V12_interaction,V14_V12_interaction,V17_V10_interaction,V17_V16_interaction,Class
0,0.207971,-0.311169,-0.617801,0.090794,-0.470401,2.536347,0.239599,-0.551600,1.378155,0.025791,...,0.000000,1.691143,6.800909,1.042926,-0.064714,-0.128485,0.192241,0.018883,-0.097830,0
1,-0.114805,-0.143772,1.065235,-0.166974,0.463917,0.166480,-0.078803,1.612727,0.448154,-0.183361,...,0.000000,0.030405,0.122273,-0.149892,0.016506,-0.122294,-0.153151,0.019169,-0.053260,0
2,1.109969,-0.165946,0.066084,0.207643,-2.890083,1.773209,0.791461,0.624501,0.379780,-0.121359,...,0.000012,4.279965,17.211818,6.592415,-0.184195,0.073351,-0.010966,0.230477,-3.207904,0
3,-0.684093,-0.287924,0.178228,-0.054952,-1.059647,1.792993,0.237609,-0.226487,-0.863291,1.965775,...,0.000012,1.395911,5.613636,-3.300273,0.196967,-0.121925,-0.051316,0.037592,0.724897,0
4,-0.237033,-1.119670,0.538196,0.753074,-0.451449,1.548718,0.592941,-0.822843,0.403034,-0.038195,...,0.000023,0.791092,3.181364,-1.010363,0.265399,-0.127570,-0.602601,-0.178504,0.107008,0


## 3. Load Feature Metadata

Feature selection should stay connected to the feature-engineering outputs, so this section reloads the final feature list and the category labels created earlier.


In [3]:
feature_columns = json.loads(FEATURE_COLUMNS_FILE.read_text(encoding="utf-8"))
feature_category_path = PROJECT_ROOT / "reports" / "tables" / "08_feature_engineering" / "final_feature_categories.json"

if feature_category_path.exists():
    feature_category_map = json.loads(feature_category_path.read_text(encoding="utf-8"))
else:
    print("WARNING: feature_category_map not found — all categories labeled UNKNOWN")
    feature_category_map = {feature: "UNKNOWN" for feature in feature_columns}

available_features = [feature for feature in feature_columns if feature in df.columns]
feature_metadata = pd.DataFrame({
    "feature": available_features,
    "feature_category": [feature_category_map.get(feature, "UNKNOWN") for feature in available_features],
})
feature_metadata.to_csv(NOTEBOOK_TABLES_DIR / "feature_metadata_overview.csv", index=False)
feature_metadata


,feature,feature_category
0,V17,PCA
1,V14,PCA
2,V12,PCA
3,V10,PCA
4,V16,PCA
5,V3,PCA
6,V7,PCA
7,V11,PCA
8,V4,PCA
9,V18,PCA


## 4. Measure Feature Relevance

This section uses simple, interpretable signals to answer whether a feature looks useful before we think about redundancy:

- correlation with `Class`
- fraud vs non-fraud mean difference
- standardized separation between the two classes

These measures are not the final modeling decision, but they give us a strong first filter.


In [4]:
relevance_rows = []
base_fraud_rate = df[TARGET_COLUMN].mean()

for feature in available_features:
    corr = df[feature].corr(df[TARGET_COLUMN])
    class_0 = df.loc[df[TARGET_COLUMN] == 0, feature]
    class_1 = df.loc[df[TARGET_COLUMN] == 1, feature]
    pooled_std = df[feature].std()
    standardized_mean_gap = abs(class_1.mean() - class_0.mean()) / pooled_std if pooled_std and not np.isnan(pooled_std) else 0.0
    relevance_rows.append({
        "feature": feature,
        "feature_category": feature_category_map.get(feature, "UNKNOWN"),
        "correlation_with_class": corr,
        "abs_correlation_with_class": abs(corr),
        "mean_class_0": class_0.mean(),
        "mean_class_1": class_1.mean(),
        "standardized_mean_gap": standardized_mean_gap,
        "base_fraud_rate": base_fraud_rate,
    })

feature_relevance_summary = pd.DataFrame(relevance_rows).sort_values(
    ["abs_correlation_with_class", "standardized_mean_gap"],
    ascending=False,
)
feature_relevance_summary.to_csv(NOTEBOOK_TABLES_DIR / "feature_relevance_summary.csv", index=False)
feature_relevance_summary.head(15)


,feature,feature_category,correlation_with_class,abs_correlation_with_class,mean_class_0,mean_class_1,standardized_mean_gap,base_fraud_rate
19,V14_V12_interaction,INTERACTION,0.571907,0.571907,-0.105309,57.377765,14.018633,0.001667
21,V17_V16_interaction,INTERACTION,0.531441,0.531441,-0.092663,51.489056,13.026740,0.001667
18,V17_V12_interaction,INTERACTION,0.529994,0.529994,-0.121215,66.126179,12.991272,0.001667
17,V17_V14_interaction,INTERACTION,0.528260,0.528260,-0.113395,61.384720,12.948752,0.001667
20,V17_V10_interaction,INTERACTION,0.518306,0.518306,-0.110313,61.775103,12.704762,0.001667
0,V17,PCA,-0.313498,0.313498,0.010963,-6.463285,7.684501,0.001667
1,V14,PCA,-0.293375,0.293375,0.011668,-6.835946,7.191247,0.001667
16,amount_V17_interaction,RATIO,-0.283188,0.283188,0.059174,-20.342903,6.941524,0.001667
2,V12,PCA,-0.250711,0.250711,0.009476,-6.103254,6.145458,0.001667
3,V10,PCA,-0.206971,0.206971,0.007663,-5.453274,5.073300,0.001667


## 5. Check Redundancy Among Features

Highly correlated engineered features can tell us the same story in slightly different ways. This section identifies feature pairs with strong overlap so we can avoid carrying duplicate signal into modeling.


In [ ]:
feature_corr_matrix = df[available_features].corr().abs()
upper_mask = np.triu(np.ones(feature_corr_matrix.shape), k=1).astype(bool)
upper_triangle = feature_corr_matrix.where(upper_mask)

redundancy_pairs = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_feature_correlation"})
    .sort_values("abs_feature_correlation", ascending=False)
)

REDUNDANCY_THRESHOLD = 0.85
flagged_redundancy_pairs = redundancy_pairs.loc[
    redundancy_pairs["abs_feature_correlation"] >= REDUNDANCY_THRESHOLD
].copy()
flagged_redundancy_pairs.to_csv(NOTEBOOK_TABLES_DIR / "feature_redundancy_pairs.csv", index=False)

redundancy_summary_rows = []
for feature in available_features:
    partner_series = upper_triangle[feature].dropna() if feature in upper_triangle.columns else pd.Series(dtype=float)
    reverse_partner_series = upper_triangle.loc[feature].dropna() if feature in upper_triangle.index else pd.Series(dtype=float)
    combined = pd.concat([partner_series, reverse_partner_series])
    if combined.empty:
        max_corr = 0.0
        strongest_partner = None
    else:
        strongest_partner = combined.idxmax()
        max_corr = combined.max()
    redundancy_summary_rows.append({
        "feature": feature,
        "strongest_partner": strongest_partner,
        "max_abs_feature_correlation": max_corr,
        "redundancy_flag": bool(max_corr >= REDUNDANCY_THRESHOLD),
    })

feature_redundancy_summary = pd.DataFrame(redundancy_summary_rows)
feature_redundancy_summary.to_csv(NOTEBOOK_TABLES_DIR / "feature_redundancy_summary.csv", index=False)
flagged_redundancy_pairs.head(15)


,feature_a,feature_b,abs_feature_correlation
323,amount_to_mean_ratio,amount_to_median_ratio,1.000000
256,Amount,amount_to_mean_ratio,1.000000
257,Amount,amount_to_median_ratio,1.000000
417,V17_V12_interaction,V17_V16_interaction,0.944739
392,V17_V14_interaction,V17_V12_interaction,0.928293
416,V17_V12_interaction,V17_V10_interaction,0.913919
461,V17_V10_interaction,V17_V16_interaction,0.892667
395,V17_V14_interaction,V17_V16_interaction,0.885799
415,V17_V12_interaction,V14_V12_interaction,0.872273
393,V17_V14_interaction,V14_V12_interaction,0.858974


## 6. Add Model-Based Feature Signals

Statistical relevance is helpful, but feature selection should also ask whether simple models continue to find the same features important. This section adds two lightweight signals:

- Logistic Regression for linear separability after scaling
- Random Forest for nonlinear importance

The goal is not to finish modeling yet. It is to make the selection stage a little more model-aware.


In [6]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    X = df[available_features]
    y = df[TARGET_COLUMN]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    logistic_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]
    )
    logistic_pipeline.fit(X_train, y_train)
    logistic_abs_coef = np.abs(logistic_pipeline.named_steps["model"].coef_[0])

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    rf_model.fit(X_train, y_train)
    rf_importance = rf_model.feature_importances_

    model_feature_signal_summary = pd.DataFrame({
        "feature": available_features,
        "logistic_abs_coef": logistic_abs_coef,
        "random_forest_importance": rf_importance,
    })
except ModuleNotFoundError as exc:
    print(f"WARNING: model-based feature signals skipped because a dependency is missing: {exc}")
    model_feature_signal_summary = pd.DataFrame({
        "feature": available_features,
        "logistic_abs_coef": np.nan,
        "random_forest_importance": np.nan,
    })

model_feature_signal_summary = model_feature_signal_summary.sort_values(
    ["random_forest_importance", "logistic_abs_coef"],
    ascending=False,
)
model_feature_signal_summary.to_csv(NOTEBOOK_TABLES_DIR / "model_feature_signal_summary.csv", index=False)
model_feature_signal_summary.head(15)


,feature,logistic_abs_coef,random_forest_importance
1,V14,1.092470,0.173060
19,V14_V12_interaction,0.640694,0.120096
21,V17_V16_interaction,0.026140,0.114861
3,V10,0.392305,0.114056
2,V12,0.931056,0.079822
18,V17_V12_interaction,0.226374,0.069850
8,V4,1.168306,0.066136
17,V17_V14_interaction,0.095018,0.049393
20,V17_V10_interaction,0.003882,0.032461
7,V11,0.347065,0.031365


## 7. Build Feature Selection Decision Table

The notebook now combines three angles:

- relevance to fraud
- redundancy with other features
- model-based usefulness

This creates an explicit keep or drop decision for each feature so the next notebook does not need to guess what belongs in the modeling dataset.


In [7]:
selection_df = (
    feature_relevance_summary
    .merge(feature_redundancy_summary, on="feature", how="left")
    .merge(model_feature_signal_summary, on="feature", how="left")
)

score_columns = [
    "abs_correlation_with_class",
    "standardized_mean_gap",
    "logistic_abs_coef",
    "random_forest_importance",
]

for column in score_columns:
    rank_column = f"{column}_rank"
    selection_df[rank_column] = selection_df[column].rank(pct=True, ascending=True)

selection_df["combined_selection_score"] = selection_df[[f"{column}_rank" for column in score_columns]].mean(axis=1, skipna=True)
selection_df["selection_reason"] = "Retained as a candidate until final rule evaluation."
selection_df["selection_decision"] = "KEEP_MONITOR"

score_lookup = selection_df.set_index("feature")["combined_selection_score"].to_dict()
redundancy_drop_map = {}
for row in flagged_redundancy_pairs.itertuples(index=False):
    score_a = score_lookup.get(row.feature_a, 0.0)
    score_b = score_lookup.get(row.feature_b, 0.0)
    drop_feature = row.feature_a if score_a < score_b else row.feature_b
    keep_feature = row.feature_b if drop_feature == row.feature_a else row.feature_a
    existing = redundancy_drop_map.get(drop_feature)
    if existing is None or row.abs_feature_correlation > existing["abs_feature_correlation"]:
        redundancy_drop_map[drop_feature] = {
            "partner": keep_feature,
            "abs_feature_correlation": row.abs_feature_correlation,
        }

for idx, row in selection_df.iterrows():
    feature = row["feature"]
    score = row["combined_selection_score"]
    abs_corr = row["abs_correlation_with_class"]
    std_gap = row["standardized_mean_gap"]

    if feature in redundancy_drop_map:
        partner = redundancy_drop_map[feature]["partner"]
        selection_df.at[idx, "selection_decision"] = "DROP_REDUNDANCY"
        selection_df.at[idx, "selection_reason"] = (
            f"Highly correlated with {partner} and carries the weaker combined signal."
        )
    elif score >= 0.65 or abs_corr >= 0.08 or std_gap >= 0.5:
        selection_df.at[idx, "selection_decision"] = "KEEP"
        selection_df.at[idx, "selection_reason"] = "Shows strong fraud relevance and remains competitive after redundancy review."
    elif score >= 0.45 or abs_corr >= 0.03:
        selection_df.at[idx, "selection_decision"] = "KEEP_MONITOR"
        selection_df.at[idx, "selection_reason"] = "Useful enough to test in baseline models, but worth monitoring for stability."
    else:
        selection_df.at[idx, "selection_decision"] = "DROP_WEAK"
        selection_df.at[idx, "selection_reason"] = "Weak relative signal across statistical and model-based screens."

selection_df = selection_df.sort_values(
    ["selection_decision", "combined_selection_score", "abs_correlation_with_class"],
    ascending=[True, False, False],
)
selection_df.to_csv(NOTEBOOK_TABLES_DIR / "feature_selection_decisions.csv", index=False)
selection_df[[
    "feature",
    "feature_category",
    "abs_correlation_with_class",
    "standardized_mean_gap",
    "max_abs_feature_correlation",
    "logistic_abs_coef",
    "random_forest_importance",
    "combined_selection_score",
    "selection_decision",
    "selection_reason",
]].head(20)


,feature,feature_category,abs_correlation_with_class,standardized_mean_gap,max_abs_feature_correlation,logistic_abs_coef,random_forest_importance,combined_selection_score,selection_decision,selection_reason
2,V17_V12_interaction,INTERACTION,0.529994,12.991272,0.944739,0.226374,0.069850,0.761364,DROP_REDUNDANCY,Highly correlated with V17_V16_interaction and...
3,V17_V14_interaction,INTERACTION,0.528260,12.948752,0.928293,0.095018,0.049393,0.704545,DROP_REDUNDANCY,Highly correlated with V17_V12_interaction and...
4,V17_V10_interaction,INTERACTION,0.518306,12.704762,0.913919,0.003882,0.032461,0.579545,DROP_REDUNDANCY,Highly correlated with V17_V12_interaction and...
7,amount_V17_interaction,RATIO,0.283188,6.941524,0.853777,0.254529,0.012969,0.556818,DROP_REDUNDANCY,Highly correlated with V17 and carries the wea...
19,amount_to_mean_ratio,RATIO,0.005777,0.141607,1.000000,0.012494,0.007533,0.136364,DROP_REDUNDANCY,Highly correlated with amount_to_median_ratio ...
21,Amount,RAW,0.005777,0.141607,1.000000,0.012494,0.007566,0.096591,DROP_REDUNDANCY,Highly correlated with amount_to_mean_ratio an...
16,time_day_fraction,NORMALIZED,0.016696,0.409247,0.416127,0.507279,0.006833,0.352273,DROP_WEAK,Weak relative signal across statistical and mo...
17,Time,RAW,0.012359,0.302953,0.422054,0.260950,0.007036,0.272727,DROP_WEAK,Weak relative signal across statistical and mo...
18,log_amount,LOG,0.007798,0.191143,0.551830,0.077377,0.008713,0.238636,DROP_WEAK,Weak relative signal across statistical and mo...
20,amount_to_median_ratio,RATIO,0.005777,0.141607,1.000000,0.012494,0.008473,0.142045,DROP_WEAK,Weak relative signal across statistical and mo...


### Redundancy Group Resolution

Pairwise correlation flags possible overlap, but the final selection rule should resolve overlap at the group level. The table below converts the flagged redundancy pairs into explicit high-correlation groups and states which features are kept versus dropped inside each group.

This makes the decision transparent. For example, the strongly overlapping `V17_*` interaction cluster no longer appears as a loose set of correlated pairs; it is resolved into a concrete keep/drop outcome.


In [10]:
from collections import defaultdict

adjacency = defaultdict(set)
for row in flagged_redundancy_pairs.itertuples(index=False):
    adjacency[row.feature_a].add(row.feature_b)
    adjacency[row.feature_b].add(row.feature_a)

visited = set()
redundancy_groups = []
for feature in sorted(adjacency):
    if feature in visited:
        continue
    stack = [feature]
    component = []
    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        component.append(current)
        stack.extend(sorted(adjacency[current] - visited))
    redundancy_groups.append(sorted(component))

decision_lookup = selection_df.set_index("feature")
group_rows = []
for group_id, group_features in enumerate(redundancy_groups, start=1):
    keep_features = []
    drop_features = []
    for feature in group_features:
        decision = decision_lookup.at[feature, "selection_decision"]
        if decision == "KEEP":
            keep_features.append(feature)
        else:
            drop_features.append(f"{feature} ({decision})")
    if keep_features and drop_features:
        group_resolution = f"Keep {', '.join(keep_features)}; drop {', '.join(drop_features)}."
    elif keep_features:
        group_resolution = f"Keep {', '.join(keep_features)}."
    else:
        group_resolution = f"Drop the entire group: {', '.join(drop_features)}."

    group_rows.append({
        "group_id": group_id,
        "group_features": ", ".join(group_features),
        "keep_features": ", ".join(keep_features),
        "drop_features": ", ".join(drop_features),
        "group_resolution": group_resolution,
    })

redundancy_group_decisions = pd.DataFrame(group_rows)
redundancy_group_decisions.to_csv(NOTEBOOK_TABLES_DIR / "redundancy_group_decisions.csv", index=False)
redundancy_group_decisions


,group_id,group_features,keep_features,drop_features,group_resolution
0,1,"Amount, amount_to_mean_ratio, amount_to_median...",,"Amount (DROP_REDUNDANCY), amount_to_mean_ratio...",Drop the entire group: Amount (DROP_REDUNDANCY...
1,2,"V14_V12_interaction, V17_V10_interaction, V17_...","V14_V12_interaction, V17_V16_interaction","V17_V10_interaction (DROP_REDUNDANCY), V17_V12...","Keep V14_V12_interaction, V17_V16_interaction;..."
2,3,"V17, amount_V17_interaction",V17,amount_V17_interaction (DROP_REDUNDANCY),Keep V17; drop amount_V17_interaction (DROP_RE...


### Amount Feature Resolution

The amount family needs an explicit final decision because `Amount`, `log_amount`, and the ratio features describe closely related transaction-size information. The table below resolves that family directly instead of leaving the reader to infer the outcome from multiple tables.

In this notebook's current results, no amount-based feature is retained in the final modeling dataset. `log_amount` is the cleaner transformed representation of raw amount, but it is still dropped because its fraud signal remains weak relative to the retained PCA and interaction features.


In [11]:
amount_feature_candidates = [
    "Amount",
    "log_amount",
    "amount_to_mean_ratio",
    "amount_to_median_ratio",
]

amount_feature_resolution = selection_df.loc[
    selection_df["feature"].isin(amount_feature_candidates),
    [
        "feature",
        "feature_category",
        "abs_correlation_with_class",
        "combined_selection_score",
        "selection_decision",
        "selection_reason",
    ],
].copy()

amount_feature_resolution["final_amount_family_decision"] = amount_feature_resolution["feature"].map({
    "log_amount": "BEST_AMOUNT_REPRESENTATION_BUT_NOT_SELECTED",
    "Amount": "DROP",
    "amount_to_mean_ratio": "DROP",
    "amount_to_median_ratio": "DROP",
})

amount_feature_resolution["amount_family_note"] = amount_feature_resolution["feature"].map({
    "log_amount": "Use this only for future sensitivity checks if one amount-based feature must be tested, but do not include it in the default selected dataset.",
    "Amount": "Raw amount is not the final choice and is excluded from the selected dataset.",
    "amount_to_mean_ratio": "This ratio is redundant with the raw amount family and is excluded from the selected dataset.",
    "amount_to_median_ratio": "This ratio is weaker than the preferred transformed amount representation and is excluded from the selected dataset.",
})

amount_feature_resolution = amount_feature_resolution.sort_values(
    ["combined_selection_score", "abs_correlation_with_class"],
    ascending=[False, False],
).reset_index(drop=True)

amount_feature_resolution.to_csv(NOTEBOOK_TABLES_DIR / "amount_feature_resolution.csv", index=False)
amount_feature_resolution


,feature,feature_category,abs_correlation_with_class,combined_selection_score,selection_decision,selection_reason,final_amount_family_decision,amount_family_note
0,log_amount,LOG,0.007798,0.238636,DROP_WEAK,Weak relative signal across statistical and mo...,BEST_AMOUNT_REPRESENTATION_BUT_NOT_SELECTED,Use this only for future sensitivity checks if...
1,amount_to_median_ratio,RATIO,0.005777,0.142045,DROP_WEAK,Weak relative signal across statistical and mo...,DROP,This ratio is weaker than the preferred transf...
2,amount_to_mean_ratio,RATIO,0.005777,0.136364,DROP_REDUNDANCY,Highly correlated with amount_to_median_ratio ...,DROP,This ratio is redundant with the raw amount fa...
3,Amount,RAW,0.005777,0.096591,DROP_REDUNDANCY,Highly correlated with amount_to_mean_ratio an...,DROP,Raw amount is not the final choice and is excl...


## 8. Build Final Selected Dataset

This section turns the decision table into a final feature selection outcome.

Only features labeled `KEEP` are exported into the default modeling dataset. Features labeled `DROP_REDUNDANCY` or `DROP_WEAK` are explicitly listed in a separate dropped-feature table. If a future run produces any `KEEP_MONITOR` features, they should stay out of the default export until they are manually promoted after model validation.


In [8]:
final_keep_features = selection_df.loc[
    selection_df["selection_decision"] == "KEEP",
    ["feature", "feature_category", "selection_decision", "selection_reason"],
].reset_index(drop=True)

final_drop_features = selection_df.loc[
    selection_df["selection_decision"] != "KEEP",
    ["feature", "feature_category", "selection_decision", "selection_reason"],
].reset_index(drop=True)

final_feature_selection_summary = (
    selection_df["selection_decision"]
    .value_counts(dropna=False)
    .rename_axis("selection_decision")
    .reset_index(name="feature_count")
)

selected_feature_names = final_keep_features["feature"].tolist()
selected_dataset = df[selected_feature_names + [TARGET_COLUMN]].copy()

final_keep_features.to_csv(NOTEBOOK_TABLES_DIR / "selected_feature_list.csv", index=False)
final_drop_features.to_csv(NOTEBOOK_TABLES_DIR / "dropped_feature_list.csv", index=False)
final_feature_selection_summary.to_csv(NOTEBOOK_TABLES_DIR / "final_feature_selection_summary.csv", index=False)
selected_dataset.to_csv(SELECTED_DATA_FILE, index=False)
SELECTED_FEATURES_FILE.write_text(json.dumps(selected_feature_names, indent=2), encoding="utf-8")

print(f"Selected feature count: {len(selected_feature_names)}")
print(f"Dropped feature count: {len(final_drop_features)}")
print(f"Selected dataset shape: {selected_dataset.shape}")
final_keep_features


Selected feature count: 12
Selected dataset shape: (283726, 13)


,feature,feature_category,selection_decision,selection_reason
0,V14_V12_interaction,INTERACTION,KEEP,Shows strong fraud relevance and remains compe...
1,V14,PCA,KEEP,Shows strong fraud relevance and remains compe...
2,V17_V16_interaction,INTERACTION,KEEP,Shows strong fraud relevance and remains compe...
3,V12,PCA,KEEP,Shows strong fraud relevance and remains compe...
4,V17,PCA,KEEP,Shows strong fraud relevance and remains compe...
5,V10,PCA,KEEP,Shows strong fraud relevance and remains compe...
6,V4,PCA,KEEP,Shows strong fraud relevance and remains compe...
7,V16,PCA,KEEP,Shows strong fraud relevance and remains compe...
8,V3,PCA,KEEP,Shows strong fraud relevance and remains compe...
9,V11,PCA,KEEP,Shows strong fraud relevance and remains compe...


## 9. Connection to Modeling and Decision System

Feature selection is the bridge between exploratory analysis and the first real modeling notebook. The decision table created here makes the next stage simpler:

- strong retained features can support the fraud probability score directly
- the final exported dataset contains only retained features, which removes ambiguity before baseline modeling starts
- dropped features stay documented, which keeps the modeling pipeline explainable and reproducible

This also helps the future decision system because `BLOCK`, `REVIEW`, and `APPROVE` logic should rely on features that remain stable after redundancy checks and basic model validation.


## 10. Key Insights

- Feature selection should reward signal quality, not just feature quantity.
- Redundant engineered features can make a model harder to interpret without adding much predictive value.
- Model-aware checks help confirm whether a statistically interesting feature still matters once we approach the modeling stage.
- The final selected dataset should be treated as the default input for baseline model comparison.


## 11. Next Step

The next notebook should build baseline fraud-detection models on the selected feature set, compare threshold-sensitive metrics, and begin defining practical `BLOCK`, `REVIEW`, and `APPROVE` cutoffs.


In [9]:
feature_selection_report = f"""# Feature Selection Report

## Key Findings

- This notebook combines univariate relevance, redundancy checks, and lightweight model-based signals to decide which engineered features move into modeling.
- Features are not selected by correlation alone; overlap between features is reviewed so the final set stays informative without unnecessary duplication.
- Logistic Regression and Random Forest are used as early model-aware checks to see whether the same features remain important once we move closer to modeling.
- The final output is a modeling-ready selected dataset plus an explicit keep or drop decision for every engineered feature.

## Feature Selection Logic

- Relevance is measured through correlation with `Class` and standardized fraud vs non-fraud separation.
- Redundancy is flagged through pairwise absolute feature correlation so duplicate signals can be pruned.
- Model-based evidence is added using Logistic Regression coefficients and Random Forest importances.
- Features are assigned to `KEEP`, `KEEP_MONITOR`, `DROP_REDUNDANCY`, or `DROP_WEAK`, but only `KEEP` features are exported into the default modeling dataset.

## Connection to Modeling and Decision System

- Retained features form the default input space for baseline fraud models.
- Any future `KEEP_MONITOR` features should remain outside the default export until a later validation step promotes them.
- Removing redundant or weak features makes downstream model behavior easier to explain and calibrate.
- A cleaner feature set supports more stable `BLOCK`, `REVIEW`, and `APPROVE` rules in the later decision system.

## Saved Tables

- `reports/tables/10_feature_selection/feature_metadata_overview.csv`
- `reports/tables/10_feature_selection/feature_relevance_summary.csv`
- `reports/tables/10_feature_selection/feature_redundancy_pairs.csv`
- `reports/tables/10_feature_selection/feature_redundancy_summary.csv`
- `reports/tables/10_feature_selection/model_feature_signal_summary.csv`
- `reports/tables/10_feature_selection/feature_selection_decisions.csv`
- `reports/tables/10_feature_selection/selected_feature_list.csv`
- `reports/tables/10_feature_selection/dropped_feature_list.csv`
- `reports/tables/10_feature_selection/final_feature_selection_summary.csv`
- `reports/tables/10_feature_selection/feature_selection_report.md`

## Saved Artifacts

- `data/processed/creditcard_selected_features.csv`
- `artifacts/selected_feature_names.json`
"""

report_path = NOTEBOOK_TABLES_DIR / "feature_selection_report.md"
report_path.write_text(feature_selection_report, encoding="utf-8")
print(f"Feature selection report saved to: {report_path}")


Feature selection report saved to: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/10_feature_selection/feature_selection_report.md
